This Jupyter notebook displays sample code used in the calculation of the example metrics in the strict and generic case on 5 points in Appendix B of [1]. The dataset on 6 Points can be found on Zenodo [2]. An excerpt of this code is part of the PhD thesis [3]. 


The following are the packages we need for the computations. Note that since there are methods implemented by multiple packages (like $\text{\texttt{OSCAR}}$ and $\text{\texttt{Graphs}}$), if loaded simultaneously the package needs to be added as a prefix to the method (e.g. $\text{\texttt{Graphs.nv()}}$). 

In [1]:
using Oscar
using Graphs
using CountingChambers

  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.7.2
─────────────────────────────────────────────────────────────────────────────
Loading images 1.3.3 (Minimal and Canonical images)
by Christopher Jefferson (http://caj.host.cs.st-andrews.ac.uk/),
   Markus Pfeiffer (http://www.morphism.de/~markusp/),
   Rebecca Waldecker (http://conway1.mathematik.uni-halle.de/~waldecker/), and
   Eliza Jonauskyte (ej31@st-andrews.ac.uk).
maintained by:
   Christopher Jefferson (http://caj.host.cs.st-andrews.ac.uk/).
Homepage: https://gap-packages.github.io/images/
Report issues at https://github.com/gap-packages/images/issues
────────────────────────────────────────────────────────────────────────────

The basic methods to enumerate the chambers of the arrangement are listed below. Their purposes are as follows: 

<dl>
<dt>$\text{\texttt{AllCycles}}$</dt>
<dd>A method to enumerate all cycles of a given graph $G$.</dd>
<dt>$\text{\texttt{WasserteinMatroid}}$</dt>
<dd>A method constructing the matroid consisting of all cycles of even length of the complete graph for a given number of vertices.</dd>
<dt>$\text{\texttt{KRWPolytope}}$</dt> 
<dd>A method to construct the KRW-polytope of a given metric.<dd>    
<dt>$\text{\texttt{MetricCone}}$</dt>   
<dd>A method to encode the metric cone (see [1], Section 2) <dd>
</dl>


In [2]:
function AllCycles(G)
    DG = SimpleDiGraph(Graphs.nv(G))
    for e in Graphs.edges(G)
        Graphs.add_edge!(DG, e.dst, e.src)
        Graphs.add_edge!(DG, e.src, e.dst)
    end
    cycs = filter(k->length(k)>2,simplecycles(DG))
    return cycs
end

AllCycles (generic function with 1 method)

In [3]:
function WassersteinMatroid(dim, return_matroid)
    G = Graphs.complete_graph(dim)
    cyclelist = AllCycles(G) 
    evencycls = filter!(x-> length(x)% 2 == 0, cyclelist) 
    listecycles = [] 
    edgelist = collect(combinations(1:dim, 2))
    for cycle in evencycls 
        permvect = vcat([1], reverse(2:length(cycle)))
        if !(cycle[permvect] in listecycles) 
            push!(listecycles, cycle)
        end 
    end 
    mat = zeros(Int, length(edgelist), length(listecycles))
    counter = 1
    for cycle in listecycles
        paritybit = 1 
        m = length(cycle)
        for i in (1:m)
            if i < m
                pair = sort(cycle[[i,i+1]]) 
            else 
                pair = sort(cycle[[1,m]]) 
            end 
            index = findfirst(x-> x==pair , edgelist)  
            mat[index, counter] = paritybit
            paritybit = paritybit * (-1)
        end 
        counter = counter + 1
    end 
    if return_matroid
        return  [matrix(Int, mat), matroid_from_matrix_columns(matrix(QQ, mat))] 
    else 
        return mat 
    end
end 

WassersteinMatroid (generic function with 1 method)

The $\text{\texttt{return\_matroid}}$ parameter lets us choose between the matroid itself, needed for subsequent methods, and the matrix underlying it to be able to actually see its structure. 

In [4]:
mat5 = WassersteinMatroid(5, false)

10×15 Matrix{Int64}:
  1   1   1   1   1   1   0   0   0   0   0   0   0   0   0
  0   0  -1   0  -1   0   1   1   1   1   0   0   0   0   0
 -1   0   0   0   0  -1  -1   0   0  -1   1   1   0   0   0
  0  -1   0  -1   0   0   0  -1  -1   0  -1  -1   0   0   0
 -1  -1   0   0   0   0  -1  -1   0   0   0   0   1   1   0
  0   0  -1  -1   0   0   1   0   0   0  -1   0   0  -1   1
  0   0   0   0  -1  -1   0   1   0   0   1   0  -1   0  -1
  1   0   1   0   0   0   0   0  -1   0   0  -1  -1   0  -1
  0   1   0   0   1   0   0   0   0  -1   0   1   0  -1   1
  0   0   0   1   0   1   0   0   1   1   0   0   1   1   0

We can use the $\text{\texttt{CountingChambers}}$ package to get the number of chambers: 

In [5]:
number_of_chambers(mat5)

882

The following methods are used to contruct the KRW polytope by first constructing the matrix of the vectors whose convex hull forms the polytope. Since we employ methods from the $\text{\texttt{OSCAR}}$ package and of $\text{\texttt{polymake}}$, we need the object encoded in both these languages. 

In [6]:
function KRWPolytopeMatrix(metric, n)
    mat = zeros(QQ, n*(n-1),n)
    counter = 1
    for i in combinations(1:n, 2)
        if ndims(metric) == 1
            value = metric[counter]
        elseif ndims(metric) == 2
            value = metric[i[1], i[2]]
        end
        mat[counter*2-1, i[1]] = 1//value
        mat[counter*2-1, i[2]] = -1//value
        mat[counter*2,i[1]] = -mat[counter*2-1, i[1]]
        mat[counter*2,i[2]] = -mat[counter*2-1, i[2]]
        counter = counter + 1
    end
    return mat
end

function KRWPolytopeOscar(metric, n)
    mat = KRWPolytopeMatrix(metric, n)
    return convex_hull(mat)
end

function KRWPolytopePolymake(metric, n)
    mat = KRWPolytopeMatrix(metric, n)
    m = size(mat,1)
    one_vector = ones(Int,m,1)
    mat = hcat(one_vector,mat)
    return Polymake.polytope.Polytope(POINTS=mat)
end

KRWPolytopePolymake (generic function with 1 method)

As an example we encode the metric with $\rho_{ij} = 1$ for all $i \not= j$. The corresponding polytope is the type $A_4$ root polytope, which we can compute with the $\text{\texttt{WasserteinPolytope}}$ method.

In [7]:
n = [0 1 1 1 ; 1 0 1 1 ; 1 1 0 1 ; 1 1 1 0]; 
P = KRWPolytopeOscar(n, 4)

Polyhedron in ambient dimension 4

In [8]:
function MetricConeMatrix(n)
    @assert n > 1
    edges = collect(combinations(1:n, 2)) 
    m = length(edges)
    mat = zeros(Int,m,m)
    for i in 1:m
        mat[i,i] = 1
    end
    S3 = symmetric_group(3)
    for tuple in combinations(1:n, 3)
        for pi in S3
            triple = [tuple[pi(i)] for i in 1:3]
            vect = repeat([0], outer = m)
            vect[findfirst(x -> x == sort([triple[1], triple[2]]), edges)] = 1 
            vect[findfirst(x -> x == sort([triple[2], triple[3]]), edges)] = 1 
            vect[findfirst(x -> x == sort([triple[1], triple[3]]), edges)] = -1 
            mat = hcat(mat, vect)
        end 
    end 
    mat = transpose(mat)
    return return mat
end 

function MetricCone(n)
    @assert n > 1
    mat = MetricConeMatrix(n)
    cone = Polymake.polytope.Cone(INEQUALITIES= mat)
    return cone
end 

MetricCone (generic function with 1 method)

The main method $\text{\texttt{getUniqueGenericMetrics}}$ constructs a hyperplane arrangement with the Wasserstein matroid as underlying matroid, finds an inner point in each chamber to construct the polytope and finally returns a list of unique elements. This is the way of enumerating the different unlabeled types. This method is very resource-intensive and thus we were only able to use it for $n \leq 5$, the computations fo $n = 6$ were done with Jörg Rambau's Software $\text{\texttt{TOPCOM}}$ [4]. The $\text{\texttt{polish\_metric}}$ method clears the denomiators from the metrics that arise as rational numbers. This improves readability of sample metrics without modifiying their combinatorial type. 

In [9]:
function polish_metric(metric)
    oscar_metric = [QQ(d) for d in metric]
    oscar_metric = lcm([denominator(i) for i in oscar_metric])*oscar_metric
end

function get_non_isomorphic_polytopes(n, metric_list)
    polytope_list = []
    for metric in metric_list 
        push!(polytope_list, KRWPolytopePolymake(metric, n))
    end
    unique_indices = []
    for i in 1:length(metric_list)
        krw_polytope = polytope_list[i]
        unique = true
        for j in 1:length(unique_indices)
            unique_krw_polytope = polytope_list[unique_indices[j]]
            if Oscar.Polymake.polytope.isomorphic(krw_polytope, unique_krw_polytope)
                unique = false
                break 
            end
        end
        if unique
            push!(unique_indices, i)
        end 
    end
    return [(polyhedron(polytope_list[i[1]]), polish_metric(metric_list[i[1]])) for i in unique_indices]
end

function getUniqueGenericMetrics(n)
    HA = Polymake.fan.HyperplaneArrangement(
     HYPERPLANES=transpose(WassersteinMatroid(n,false)), 
     SUPPORT = MetricCone(n))
    decomp = HA.CHAMBER_DECOMPOSITION
    m = decomp.MAXIMAL_CONES
    raymatrix = decomp.RAYS
    metric_list = [] 
    println("number of chambers: ", decomp.N_MAXIMAL_CONES)
    for i in 1:size(m)[1]
        cone = Polymake.polytope.Cone(INPUT_RAYS=raymatrix[m[i,:], :])
        metric = cone.REL_INT_POINT
        push!(metric_list, metric)
    end 
    return get_non_isomorphic_polytopes(n, metric_list)
end


getUniqueGenericMetrics (generic function with 1 method)

In [10]:
M4 = getUniqueGenericMetrics(4)
@time M5 = getUniqueGenericMetrics(5)

number of chambers: 6
number of chambers: 882
 17.696791 seconds (154.29 M allocations: 2.063 GiB, 2.66% gc time, 0.10% compilation time)


12-element Vector{Tuple{Polyhedron{QQFieldElem}, Vector{QQFieldElem}}}:
 (Polytope in ambient dimension 5, [8, 18, 15, 11, 14, 15, 15, 7, 11, 14])
 (Polytope in ambient dimension 5, [8, 20, 15, 13, 16, 19, 13, 9, 11, 16])
 (Polytope in ambient dimension 5, [8, 18, 19, 11, 18, 15, 15, 7, 11, 14])
 (Polytope in ambient dimension 5, [10, 22, 17, 11, 16, 15, 17, 9, 15, 20])
 (Polytope in ambient dimension 5, [6, 12, 8, 6, 9, 8, 8, 6, 8, 12])
 (Polytope in ambient dimension 5, [6, 11, 7, 6, 8, 9, 6, 6, 7, 11])
 (Polytope in ambient dimension 5, [10, 22, 15, 13, 16, 21, 11, 11, 13, 20])
 (Polytope in ambient dimension 5, [12, 24, 25, 11, 24, 17, 19, 11, 17, 24])
 (Polytope in ambient dimension 5, [7, 16, 11, 10, 11, 16, 10, 7, 10, 10])
 (Polytope in ambient dimension 5, [14, 30, 21, 17, 20, 15, 27, 13, 17, 26])
 (Polytope in ambient dimension 5, [7, 13, 8, 7, 9, 7, 10, 7, 8, 13])
 (Polytope in ambient dimension 5, [11, 20, 20, 11, 20, 11, 20, 11, 11, 20])

We save the polytopes with their corresponding metric in a JSON file in the Zenodo data base [2]. 

In [31]:
save("Polytopes_generic_metrics_5.mrdi", M5)

In [32]:
M5_loaded = load("Polytopes_generic_metrics_5.mrdi") 

12-element Vector{Tuple{Polyhedron{QQFieldElem}, Vector{QQFieldElem}}}:
 (Polytope in ambient dimension 5, [8, 18, 15, 11, 14, 15, 15, 7, 11, 14])
 (Polytope in ambient dimension 5, [8, 20, 15, 13, 16, 19, 13, 9, 11, 16])
 (Polytope in ambient dimension 5, [8, 18, 19, 11, 18, 15, 15, 7, 11, 14])
 (Polytope in ambient dimension 5, [10, 22, 17, 11, 16, 15, 17, 9, 15, 20])
 (Polytope in ambient dimension 5, [6, 12, 8, 6, 9, 8, 8, 6, 8, 12])
 (Polytope in ambient dimension 5, [6, 11, 7, 6, 8, 9, 6, 6, 7, 11])
 (Polytope in ambient dimension 5, [10, 22, 15, 13, 16, 21, 11, 11, 13, 20])
 (Polytope in ambient dimension 5, [12, 24, 25, 11, 24, 17, 19, 11, 17, 24])
 (Polytope in ambient dimension 5, [7, 16, 11, 10, 11, 16, 10, 7, 10, 10])
 (Polytope in ambient dimension 5, [14, 30, 21, 17, 20, 15, 27, 13, 17, 26])
 (Polytope in ambient dimension 5, [7, 13, 8, 7, 9, 7, 10, 7, 8, 13])
 (Polytope in ambient dimension 5, [11, 20, 20, 11, 20, 11, 20, 11, 11, 20])

The symmetric group $S_n$ acts on the Wasserstein arrangement by permutation of the labels of the $n$ points of the space, which in turn induces a permutation of the labels of the distances of the metric. Thus, if the want to compute the orbit of a chamber of the arrangement and the stabilizer, we first need to encode this symmetry group. This is what the function $\text{\texttt{symmetry\_group\_in\_wasserstein}}$ is for. 

In [13]:
function map_permutation_to_Starget(n, mat, sigma)
    pairs = collect(combinations(1:n,2))
    permuted_pairs = [sort([sigma(p[1]),sigma(p[2])]) for p in pairs]
    sigma_on_pairs = perm([findfirst(x -> x==p, permuted_pairs) for p in pairs])
    
    final_perm = Vector{Int}()
    all_columns = [mat[:,k] for k in 1:size(mat,2)]
    for i in 1:size(mat,2)
        permuted_column = permuted(mat[:,i], sigma_on_pairs)
        new_index = findfirst(x -> x==permuted_column, all_columns)
        if typeof(new_index) != Int
            new_index = -1*findfirst(x -> x==-1*permuted_column, all_columns)
        end
        push!(final_perm,new_index)
    end
    abs_perm = perm([abs(i) for i in final_perm])
    # we use signed permutation matrix to account for sign flips
    pmatrix = permutation_matrix(ZZ, abs_perm)
    for i in 1:size(mat,2)
        if sign(final_perm[i]) == -1
            pmatrix[i,:] = -1*pmatrix[i,:]
        end
    end
    return pmatrix
end

function symmetry_group_in_wasserstein(n, mat)
    Sn = symmetric_group(n)
    Starget = general_linear_group(size(mat,2),ZZ)
    G = gens(Sn)
    Gtarget = [map_permutation_to_Starget(n, mat, sigma) for sigma in G]
    emb = hom(Sn, Starget, G, Gtarget)
    H = emb(Sn)
    return H[1], emb
end

symmetry_group_in_wasserstein (generic function with 1 method)

In [14]:
function matrix_act(vect, mat)
    return mat*matrix(vect)
end

function stabilizer_from_metric(n, mat, H, metric)
    sign_pattern = transpose(mat)*metric
    sign_pattern = matrix([ZZ(sign(i)) for i in sign_pattern])
    O = orbit(H, matrix_act, sign_pattern)
    println("order of the orbit ", length(O))
    return stabilizer(O)[1]
end

stabilizer_from_metric (generic function with 1 method)

As an example, we compute the generic metrics for $n = 5$. 

In [15]:
n = 5
H, emb = symmetry_group_in_wasserstein(n, mat5)
counter = 0
for i in 1:length(M5)
    println("index: ", i)
    println("metric: ", M5[i][2])
    stab = stabilizer_from_metric(5, mat5, H, M5[i][2])
    println("order of the stabilizer: ", order(stab))
    stab_n = preimage(emb,stab)[1]
    global counter += 120/order(stab_n)
    println("type of the stabilizer: ", describe(stab_n))
    println("generators of the stabilizer: ", gens(stab_n))
    println()
end
counter

index: 1
metric: QQFieldElem[8, 18, 15, 11, 14, 15, 15, 7, 11, 14]
order of the orbit 60
order of the stabilizer: 2
type of the stabilizer: C2
generators of the stabilizer: PermGroupElem[(1,3)(2,4)]

index: 2
metric: QQFieldElem[8, 20, 15, 13, 16, 19, 13, 9, 11, 16]
order of the orbit 120
order of the stabilizer: 1
type of the stabilizer: 1
generators of the stabilizer: PermGroupElem[]

index: 3
metric: QQFieldElem[8, 18, 19, 11, 18, 15, 15, 7, 11, 14]
order of the orbit 60
order of the stabilizer: 2
type of the stabilizer: C2
generators of the stabilizer: PermGroupElem[(1,3)(2,4)]

index: 4
metric: QQFieldElem[10, 22, 17, 11, 16, 15, 17, 9, 15, 20]
order of the orbit 120
order of the stabilizer: 1
type of the stabilizer: 1
generators of the stabilizer: PermGroupElem[]

index: 5
metric: QQFieldElem[6, 12, 8, 6, 9, 8, 8, 6, 8, 12]
order of the orbit 120
order of the stabilizer: 1
type of the stabilizer: 1
generators of the stabilizer: PermGroupElem[]

index: 6
metric: QQFieldElem[6, 11,

882

The analogue methods can be used for the strict, non-generic case. We do not consider the maximal cones, but all others. 

In [25]:
function getUniqueNonGenericStrictMetrics(n)
    m = binomial(n,2)
    MC = MetricCone(n)
    matrix_of_metric_cone = MetricConeMatrix(n)
    HA = Polymake.fan.HyperplaneArrangement(HYPERPLANES=transpose(Matrix{Int64}(WassersteinMatroid(n,false))), 
        SUPPORT = MC);
    decomp = HA.CHAMBER_DECOMPOSITION
    Cs = decomp.CONES
    #println("done with cone decomposition")
    raymatrix = decomp.RAYS
    metriclist = [] 
    for i in 1:(decomp.FAN_DIM-1)
        for j in 1:size(Cs[i])[1]
            co = Polymake.polytope.Cone(INPUT_RAYS = raymatrix[Cs[i][j,:],:])
            metric = co.REL_INT_POINT
            # check if the metric is strict:
            vec = matrix_of_metric_cone*metric
            if all(x-> x > 0, vec)
                push!(metriclist, metric)
            end
        end 
    end
    #println("done with metrics. found ", length(metriclist), " strict metrics.")
    return get_non_isomorphic_polytopes(n, metriclist)
end

getUniqueNonGenericStrictMetrics (generic function with 1 method)

In [26]:
S4 = getUniqueNonGenericStrictMetrics(4) 
@time S5 = getUniqueNonGenericStrictMetrics(5)

1409.469376 seconds (4.20 G allocations: 58.290 GiB, 1.81% gc time, 0.00% compilation time)


65-element Vector{Tuple{Polyhedron{QQFieldElem}, Vector{QQFieldElem}}}:
 (Polytope in ambient dimension 5, [2, 2, 2, 2, 2, 2, 2, 2, 2, 2])
 (Polytope in ambient dimension 5, [2, 3, 3, 2, 3, 3, 2, 2, 3, 3])
 (Polytope in ambient dimension 5, [4, 6, 5, 5, 6, 5, 5, 5, 5, 6])
 (Polytope in ambient dimension 5, [3, 4, 3, 3, 3, 4, 4, 3, 3, 4])
 (Polytope in ambient dimension 5, [3, 5, 4, 4, 4, 5, 3, 3, 3, 4])
 (Polytope in ambient dimension 5, [2, 4, 4, 3, 4, 4, 3, 2, 3, 3])
 (Polytope in ambient dimension 5, [4, 8, 7, 5, 8, 7, 5, 5, 7, 8])
 (Polytope in ambient dimension 5, [4, 8, 7, 7, 8, 7, 7, 5, 5, 6])
 (Polytope in ambient dimension 5, [3, 5, 4, 3, 4, 5, 4, 3, 4, 5])
 (Polytope in ambient dimension 5, [6, 10, 7, 7, 8, 9, 9, 7, 7, 10])
 (Polytope in ambient dimension 5, [6, 10, 9, 7, 8, 7, 9, 5, 7, 8])
 (Polytope in ambient dimension 5, [3, 6, 5, 4, 5, 6, 3, 3, 4, 5])
 (Polytope in ambient dimension 5, [4, 7, 5, 5, 5, 7, 5, 4, 4, 6])
 (Polytope in ambient dimension 5, [6, 12, 9, 9, 10, 1

In [27]:
n = 5
H, emb = symmetry_group_in_wasserstein(n, mat5)
counter = 0

sorted_collection = Dict()
# [f_vector, [(metric, group)]]
for i in 1:length(S5)
    println("index: ", i)
    println("metric: ", S5[i][2])
    fvec = f_vector(S5[i][1])
    println("f vector: ", fvec)
    
    stab = stabilizer_from_metric(5, mat5, H, S5[i][2])
    println("order of the stabilizer: ", order(stab))
    stab_n = preimage(emb,stab)[1]
    global counter += 120/order(stab_n)
    println("type of the stabilizer: ", describe(stab_n))
    println("generators of the stabilizer: ", gens(stab_n))
    println()
    if haskey(sorted_collection, fvec)
        push!(sorted_collection[fvec],(S5[i][2], describe(stab_n)))
    else
        sorted_collection[fvec] = [(S5[i][2], describe(stab_n))]
    end     
end
counter

index: 1
metric: QQFieldElem[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
f vector: ZZRingElem[20, 60, 70, 30]
order of the orbit 1
order of the stabilizer: 120
type of the stabilizer: S5
generators of the stabilizer: PermGroupElem[(1,2,3,4,5), (1,2)]

index: 2
metric: QQFieldElem[2, 3, 3, 2, 3, 3, 2, 2, 3, 3]
f vector: ZZRingElem[20, 72, 94, 42]
order of the orbit 10
order of the stabilizer: 12
type of the stabilizer: D12
generators of the stabilizer: PermGroupElem[(1,2), (1,5), (3,4)]

index: 3
metric: QQFieldElem[4, 6, 5, 5, 6, 5, 5, 5, 5, 6]
f vector: ZZRingElem[20, 76, 102, 46]
order of the orbit 30
order of the stabilizer: 4
type of the stabilizer: C2 x C2
generators of the stabilizer: PermGroupElem[(1,2), (4,5)]

index: 4
metric: QQFieldElem[3, 4, 3, 3, 3, 4, 4, 3, 3, 4]
f vector: ZZRingElem[20, 72, 94, 42]
order of the orbit 10
order of the stabilizer: 12
type of the stabilizer: D12
generators of the stabilizer: PermGroupElem[(1,3)(2,4,5), (4,5)]

index: 5
metric: QQFieldElem[3, 5, 4, 4, 4, 5

5661

In [28]:
sorted_fvecs = sort(collect(keys(sorted_collection)))
f_counter = 1
for fvec in sorted_fvecs
    println("f vector: ", Vector{Int}(fvec), " has ", length(sorted_collection[fvec]), " polytopes")
    #println(length(sorted_collection[fvec]))
    inner_counter = 1
    for i in 1:length(sorted_collection[fvec])
        current_metric, group = collect(sorted_collection[fvec])[i]
        int_metric = [Int(m) for m in current_metric]
        println("index: ", f_counter, ".", inner_counter, " metric: ", int_metric, " group: ", group )
        inner_counter += 1
    end
    println()
    f_counter += 1
end

f vector: [20, 60, 70, 30] has 1 polytopes
index: 1.1 metric: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2] group: S5

f vector: [20, 72, 94, 42] has 2 polytopes
index: 2.1 metric: [2, 3, 3, 2, 3, 3, 2, 2, 3, 3] group: D12
index: 2.2 metric: [3, 4, 3, 3, 3, 4, 4, 3, 3, 4] group: D12

f vector: [20, 76, 102, 46] has 1 polytopes
index: 3.1 metric: [4, 6, 5, 5, 6, 5, 5, 5, 5, 6] group: C2 x C2

f vector: [20, 80, 112, 52] has 3 polytopes
index: 4.1 metric: [4, 8, 7, 5, 8, 7, 5, 5, 7, 8] group: C2
index: 4.2 metric: [3, 5, 4, 3, 4, 5, 4, 3, 4, 5] group: C2
index: 4.3 metric: [6, 10, 7, 7, 8, 9, 9, 7, 7, 10] group: C2

f vector: [20, 80, 114, 54] has 4 polytopes
index: 5.1 metric: [2, 4, 4, 3, 4, 4, 3, 2, 3, 3] group: D8
index: 5.2 metric: [4, 8, 7, 7, 8, 7, 7, 5, 5, 6] group: C2 x C2
index: 5.3 metric: [3, 4, 3, 3, 4, 3, 3, 3, 3, 4] group: C2 x C2
index: 5.4 metric: [10, 14, 9, 9, 10, 11, 11, 9, 9, 14] group: D8

f vector: [20, 82, 116, 54] has 1 polytopes
index: 6.1 metric: [3, 5, 4, 4, 4, 5, 3, 3, 3, 4

We save the polytopes with their corresponding metircs in a JSON file in the Zenodo data base [2]. 

In [29]:
save("Polytopes_strict_metrics_5.mrdi",S5)

In [30]:
load("Polytopes_strict_metrics_5.mrdi")

65-element Vector{Tuple{Polyhedron{QQFieldElem}, Vector{QQFieldElem}}}:
 (Polytope in ambient dimension 5, [2, 2, 2, 2, 2, 2, 2, 2, 2, 2])
 (Polytope in ambient dimension 5, [2, 3, 3, 2, 3, 3, 2, 2, 3, 3])
 (Polytope in ambient dimension 5, [4, 6, 5, 5, 6, 5, 5, 5, 5, 6])
 (Polytope in ambient dimension 5, [3, 4, 3, 3, 3, 4, 4, 3, 3, 4])
 (Polytope in ambient dimension 5, [3, 5, 4, 4, 4, 5, 3, 3, 3, 4])
 (Polytope in ambient dimension 5, [2, 4, 4, 3, 4, 4, 3, 2, 3, 3])
 (Polytope in ambient dimension 5, [4, 8, 7, 5, 8, 7, 5, 5, 7, 8])
 (Polytope in ambient dimension 5, [4, 8, 7, 7, 8, 7, 7, 5, 5, 6])
 (Polytope in ambient dimension 5, [3, 5, 4, 3, 4, 5, 4, 3, 4, 5])
 (Polytope in ambient dimension 5, [6, 10, 7, 7, 8, 9, 9, 7, 7, 10])
 (Polytope in ambient dimension 5, [6, 10, 9, 7, 8, 7, 9, 5, 7, 8])
 (Polytope in ambient dimension 5, [3, 6, 5, 4, 5, 6, 3, 3, 4, 5])
 (Polytope in ambient dimension 5, [4, 7, 5, 5, 5, 7, 5, 4, 4, 6])
 (Polytope in ambient dimension 5, [6, 12, 9, 9, 10, 1

To visualize the polytopes, one can compute the Schlegel diagrams. As noted in https://mathrepo.mis.mpg.de/OSCAR/OscarPolytopes.html, the command to visualize a Polymake polytope is $\text{\texttt{Polymake.polytope.visual(P)}}$, but this command has issues in a Jupyter notebook like this one. It can be used in the julia command line. 

$\huge{\text{References}}$ 

[1] Delucchi, E., Kühne, L., Mühlherr, L. (2025), $\text{\emph{Combinatorial invariants of finite metric spaces and the Wasserstein arrangement}}$, 2025, https://arxiv.org/abs/2408.15584. 

[2] Delucchi, E., Kühne, L., Mühlherr, L., & Rambau, J. (2024).  $\text{\emph{Dataset of Kantorovich-Rubinstein-Wasserstein Polytopes of Metric Spaces on up to 6 Points (topcom 1.1.5, Julia 1.10.2, OSCAR 1.1.1)}}$. Zenodo. https://doi.org/10.5281/zenodo.12773907

[3] Mühlherr L. (2026) $\text{\emph{A Graph Theoretical Perspective on Hyperplane Arrangement Theory}}$. Bielefeld: Universität Bielefeld. https://doi.org/10.4119/unibi/3013025 

[4] Rambau, J.(2023). $\text{\emph{Symmetric lexicographic subset reverse search for the enumeration of circuits, cocircuits, and triangulations up to symmetry.}}$ https://www.wm.uni-bayreuth.de/de/team/rambau_joerg/TOPCOMSymLexSubsetRS-2.pdf.
